# Assessing Urban Rooftop Photovoltaic Potential in Nanjing Using PyGeoModel
## 1. Background and Research Objectives
### 1.1 Introduction to Urban Rooftop Photovoltaic Potential
Rooftop photovoltaic (PV) systems represent a promising solution for addressing urban energy demands through renewable energy sources. As urban centers continue to grow, the integration of solar energy into existing infrastructure becomes increasingly important for sustainable development. Rooftop PV systems offer several advantages:
- Utilize existing infrastructure without requiring additional land
- Reduce transmission losses by generating electricity close to consumption points
- Lower carbon emissions compared to conventional energy sources
- Contribute to energy independence for urban residents and businesses

### 1.2 Research Context: Xuanwu District, Nanjing City, China
Nanjing, the capital of Jiangsu Province, serves as an ideal case study for assessing rooftop PV potential:
- Population: 8.44 million residents
- Economic significance: Major economic hub in eastern China
- Climate conditions: Favorable solar radiation with an annual average of approximately 1,370 W/m²
- Energy demand: Growing industrial and residential electricity requirements
- Sustainability goals: Aligns with China's renewable energy transition policies

### 1.3 Research Objectives
This study aims to:

- Quantify the spatial distribution of rooftop PV potential across Xuanwu district, Nanjing city
- Estimate the total potential power generation from rooftop PV systems
- Calculate potential carbon emission reductions
- Demonstrate an efficient workflow for PV potential assessment using PyGeoModel

# 2. Data Preprocessing
## 2.1 Data Sources
For this analysis, we utilize two primary datasets:
1. Xuanwu District Rooftop Vector Data
- Contains polygon geometries for building rooftops across Xuanwu District
- Includes attributes: rooftop ID, area (m²), building type, height
- Format: ESRI Shapefile (.shp)

## 2.2 Environment Dependencies

Before proceeding with the analysis, we need to import the required Python packages.

In [ ]:
# Import necessary libraries
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
from ipyleaflet import Map, GeoData, basemaps
from IPython.display import display, HTML
from pygeomodel import GeoModeler


## 2.3 Data Loading and Initial Visualization

### 2.3.1 Loading Xuanwu District Rooftop Data

Load the rooftop vector data and create an initial interactive map to visualize the study area:


In [ ]:
xuanwu_path = "data/xuanwu/xuanwu.shp"
gdf = gpd.read_file(xuanwu_path)

m = folium.Map(location=[32.079, 118.75], zoom_start=13)

folium.GeoJson(
    gdf.to_json(),
    style_function=lambda x: {
        'fillColor': 'blue',
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.5,
    }
).add_to(m)

m

### 2.3.2 Statistical Analysis of Rooftop Data

Perform comprehensive statistical analysis of the rooftop dataset to understand the distribution and characteristics of building rooftops in Xuanwu District:


In [ ]:
xuanwu_rooftops = gpd.read_file("data/xuanwu/xuanwu.shp")

total_rooftops = len(xuanwu_rooftops)
avg_area_m2 = xuanwu_rooftops["area"].mean()
total_area_m2 = xuanwu_rooftops["area"].sum()

total_area_km2 = total_area_m2 / 1e6
avg_area_km2 = avg_area_m2 / 1e6

print(f"Total number of roofs:{total_rooftops}")
print(f"Average roof area:{avg_area_m2:.2f} square meter ({avg_area_km2:.6f} square kilometer)")
print(f"Total roof area:{total_area_m2:.2f} square meter ({total_area_km2:.2f} square kilometer)")

# Draw interactive histograms using Plotly
import plotly.graph_objects as go
import numpy as np

fig = go.Figure(data=[go.Histogram(
    x=xuanwu_rooftops["area"],
    nbinsx=300,
    marker_color="#FF5733",
    opacity=0.7,
    name="frequency"
)])

fig.update_layout(
    title="Distribution of roof area in Qinhuai District",
    xaxis_title="Area (square meters)",
    yaxis_title="Frequent",
    bargap=0.1,
    template="plotly",
    showlegend=True,
    width=800,
    height=400,
    xaxis=dict(
        range=[0, 5000],
        rangeslider=dict(visible=True),
        type="linear"
    )
)

fig.show()

# 3. Photovoltaic Potential Assessment Using PyGeoModel

## 3.1 Model Initialization and Selection

Initialize the PyGeoModel framework and explore available models for rooftop photovoltaic potential assessment:


In [ ]:
modeler = GeoModeler()

recommendation = modeler.suggest_model()
recommendation


## 3.2 Model Execution

Execute the "Roof Photovoltaic Carbon Emission Reduction Potential Assessment Model" to calculate photovoltaic potential for each rooftop in the study area. Because this model processes a large volume of remote-sensing imagery over a one-year period, the online model execution may take approximately 10 minutes to complete:


In [ ]:
result = modeler.invoke(
    "Roof Photovoltaic Carbon Emission Reduction Potential Assessment Model",
    params={
        "system_efficiency": 0.8,
        "start_time": "2018-01",
        "end_time": "2018-12",
        "roof_vector_path": "data/xuanwu_rooftop.zip",
    },
)
saved_files = result.save(output_dir="./data/")
saved_files


# 4. Results Analysis and Visualization

## 4.1 Loading Model Results

Load the processed results containing power generation potential for each rooftop. These files are included in the repository so that the visualization and statistical analysis can be reproduced even when the online model-service call is not re-run:


In [ ]:
shp_file_path = "data/result/roof_results_with_power_generation.shp"
gdf = gpd.read_file(shp_file_path)

power_gene = gdf['power_gene']

## 4.2 Coordinate System Standardization

Ensure all spatial data uses the WGS84 coordinate system (EPSG:4326) for consistent mapping and visualization:


In [ ]:
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs(epsg=4326)

## 4.3 Power Generation Heatmap Creation

Create an interactive heatmap visualization showing the spatial distribution of photovoltaic power generation potential across Xuanwu District:

### 4.3.1 Data Preprocessing for Visualization

This section performs several key preprocessing steps:
1. **Coordinate Transformation**: Convert to UTM for accurate centroid calculation
2. **Centroid Calculation**: Find the geographic center of each rooftop polygon
3. **Data Normalization**: Apply logarithmic transformation and ranking to handle data skewness
4. **Heatmap Data Preparation**: Format data for Folium heatmap visualization


In [ ]:
gdf_utm = gdf.to_crs(epsg=32650)
gdf_utm['centroid'] = gdf_utm['geometry'].centroid
gdf['centroid'] = gdf_utm['centroid'].to_crs(epsg=4326)
gdf['latitude'] = gdf['centroid'].y
gdf['longitude'] = gdf['centroid'].x


power_gene_log = np.log1p(gdf['power_gene'])

power_gene_rank = power_gene_log.rank()
power_gene_rank_normalized = (power_gene_rank - power_gene_rank.min()) / (power_gene_rank.max() - power_gene_rank.min())

heat_data = [[row['latitude'], row['longitude'], power_gene_rank_normalized[i]] for i, row in gdf.iterrows()]

m = folium.Map(location=[32.04, 118.78], zoom_start=13, tiles='CartoDB Positron')

HeatMap(
    heat_data,
    radius=5,
    blur=5,
    min_opacity=0.4,
    max_opacity=0.9,
    gradient={'0.0': 'purple', '0.25': 'blue', '0.5': 'green', '0.75': 'yellow', '1.0': 'red'}
).add_to(m)

# m.save('power_generation_heatmap.html')

m

## 4.4 Results Summary

Simple statistical summary of the photovoltaic power generation analysis:

In [ ]:
# Calculate basic statistics for power generation results
total_power_generation = gdf['power_gene'].sum()
mean_power_generation = gdf['power_gene'].mean()
max_power_generation = gdf['power_gene'].max()

print("=== Power Generation Analysis Results ===")
print(f"Total Rooftops Analyzed: {len(gdf):,}")
print(f"Total Power Generation Potential: {total_power_generation:,.0f} kWh/year")
print(f"Average Power Generation per Rooftop: {mean_power_generation:.0f} kWh/year")
print(f"Maximum Power Generation: {max_power_generation:.0f} kWh/year")

# Convert to GWh for better understanding
total_power_gwh = total_power_generation / 1000000
print(f"Total Power Generation: {total_power_gwh:.2f} GWh/year")

# Calculate carbon emission reduction potential
co2_reduction_tons = (total_power_generation * 0.5) / 1000  # 0.5 kg CO2 per kWh
print(f"Potential CO2 Reduction: {co2_reduction_tons:,.0f} tons/year")


# 5. Summary and Conclusions

## Analysis Results

This study successfully assessed rooftop photovoltaic potential in Xuanwu District, Nanjing using PyGeoModel framework:

### Key Findings

- **Study Area**: 19,128 rooftops covering 11.77 km² total area in Xuanwu District
- **Total Power Generation Potential**: 2,365.28 GWh/year (2.37 TWh/year)
- **Average Generation per Rooftop**: 123,656 kWh/year
- **Maximum Single Rooftop Potential**: 9.8 GWh/year
- **Environmental Impact**: Potential CO2 reduction of 1.18 million tons/year
